# Dialog Act ClassificationConverted from `src/dialog_analysis.py`---

**Beschreibung:** Dialog Analysis - Classifies comments by communication type

In [4]:
import pandas as pd
import re
from pathlib import Path

# Dialog Act Categories (code → human-readable name)
DIALOG_ACTS = {
    'QUESTION':   'Question',
    'ANSWER':     'Answer',
    'GREETING':   'Greeting',
    'COMPLAINT':  'Complaint',
    'THANKS':     'Thanks',
    'APOLOGY':    'Apology',
    'REQUEST':    'Request',
    'INFORM':     'Information',
    'CONFIRM':    'Confirmation',
    'REJECT':     'Rejection',
    'PROMISE':    'Promise',
    'OTHER':      'Other'
}

# Regex patterns – bilingual (EN + DE)
PATTERNS = {
    'QUESTION': [
        r'\?$',                                           # ends with question mark
        r'^(how|what|when|where|why|who|can|could|is|are|do|does)\b',
        r'^(wie|was|wann|wo|warum|wer|kann|ist|sind)\b'
    ],
    'GREETING': [
        r'^(hi|hello|hey|dear|good morning|good afternoon)\b',
        r'^(hallo|guten tag|liebe|sehr geehrte)\b',
        r'(regards|best|thanks|cheers)[\s,]*$'
    ],
    'COMPLAINT': [
        r'\b(not working|broken|issue|problem|error|bug|fail|wrong)\b',
        r'\b(funktioniert nicht|kaputt|fehler|problem|falsch)\b'
    ],
    'THANKS': [
        r'\b(thank|thanks|appreciate|grateful)\b',
        r'\b(danke|vielen dank)\b'
    ],
    'APOLOGY': [
        r'\b(sorry|apolog|regret|excuse)\b',
        r'\b(entschuldigung|tut mir leid)\b'
    ],
    'REQUEST': [
        r'\b(please|could you|would you|can you|need)\b',
        r'\b(bitte|könnten sie|würden sie|brauche)\b',
        r'\b(urgent|asap|priority)\b'
    ],
    'CONFIRM': [
        r'\b(yes|correct|confirmed|ok|okay|sure)\b',
        r'\b(ja|korrekt|bestätigt|einverstanden)\b'
    ],
    'REJECT': [
        r'\b(no|cannot|unable|impossible|denied)\b',
        r'\b(nein|kann nicht|unmöglich)\b'
    ],
    'PROMISE': [
        r'\b(will|going to|promise|commit)\b',
        r'\b(werde|werden|verspreche)\b'
    ],
    'INFORM': [
        r'\b(fyi|for your information|please note|update)\b',
        r'\b(zur information|hinweis|aktualisierung)\b'
    ]
}


def classify_text(text):
    """
    Rule-based classification of a single utterance into a dialog act.
    
    Returns:
        dict with 'act', 'name', 'confidence'
    """
    if not text or not isinstance(text, str) or len(text.strip()) < 3:
        return {'act': 'OTHER', 'name': 'Other', 'confidence': 0.0}

    text = text.strip()
    matches = {}

    for act, patterns in PATTERNS.items():
        match_count = sum(1 for pat in patterns if re.search(pat, text, re.IGNORECASE))
        if match_count > 0:
            matches[act] = match_count

    if not matches:
        return {'act': 'OTHER', 'name': 'Other', 'confidence': 0.3}

    best_act = max(matches, key=matches.get)
    # Very basic confidence heuristic
    confidence = min(matches[best_act] / len(PATTERNS[best_act]), 1.0)

    return {
        'act': best_act,
        'name': DIALOG_ACTS[best_act],
        'confidence': round(confidence, 2)
    }


def process_comments(utterances_df):
    """
    Apply dialog act classification to every row in the DataFrame.
    """
    print("Classifying comments...")

    # Flexible column name detection
    text_col = 'actionbody' if 'actionbody' in utterances_df.columns else 'body'
    if text_col not in utterances_df.columns:
        raise ValueError("No text column found ('actionbody' or 'body' expected)")

    results = []
    total = len(utterances_df)

    for idx, row in utterances_df.iterrows():
        text = row.get(text_col, "")
        result = classify_text(str(text) if pd.notna(text) else "")

        results.append({
            'issueid':         row.get('issueid', idx),
            'author':          row.get('author', 'unknown'),
            'author_role':     row.get('author_role', 'unknown'),
            'dialog_act':      result['act'],
            'dialog_act_name': result['name'],
            'confidence':      result['confidence'],
            'text_preview':    str(text)[:100] + ('...' if len(str(text)) > 100 else '')
        })

        if (idx + 1) % 5000 == 0:
            print(f"   {idx+1:,} / {total:,} classified...")

    print(f" {len(results):,} comments classified")
    return pd.DataFrame(results)


def get_distribution(dialog_df):
    """Print a nice distribution of dialog acts."""
    if 'dialog_act' not in dialog_df.columns:
        print("Error: 'dialog_act' column not found in DataFrame")
        return None

    distribution = dialog_df['dialog_act'].value_counts()
    total = len(dialog_df)

    print("\nDialog Act Distribution:")
    print("-" * 50)
    for act, count in distribution.items():
        pct = count / total * 100
        name = DIALOG_ACTS.get(act, act)
        print(f"   {name:<18} {count:>6}  ({pct:5.1f}%)")
    print("-" * 50)

    return distribution

##  Execution

In [5]:
import pandas as pd
from pathlib import Path

print("=" * 50)
print(" DIALOG ACT ANALYSIS")
print("=" * 50)
print(f"Current working directory: {Path('.').absolute()}\n")

data_path = Path("data/raw/sample_utterances.csv")

if not data_path.exists():
    print("❌ File not found")
    print(f"   Path tried: {data_path.absolute()}")
    print("   Common fixes:")
    print("     • Move file to data/raw/")
    print("     • Check spelling / hidden extensions (.csv.csv)")
    print("     • Run !pwd or print(Path.cwd()) to confirm directory")
else:
    try:
        utterances = pd.read_csv(data_path)
        print(f"✓ Loaded successfully: {len(utterances):,} rows")
        
        print("\nColumns in file:")
        print(utterances.columns.tolist())
        
        text_col = 'actionbody' if 'actionbody' in utterances.columns else 'body'
        print(f"→ Using text column: {text_col}")
        print(f"   Missing/NaN texts: {utterances[text_col].isna().sum():,}")
        
        if 'issueid' not in utterances.columns:
            print("⚠️  Warning: 'issueid' column missing → using row index as fallback")
        
        # Run classification
        print("\nStarting classification...")
        dialog_df = process_comments(utterances)
        
        # Distribution
        get_distribution(dialog_df)
        
        # Export
        output_path = Path("data/processed/dialog_acts.csv")
        output_path.parent.mkdir(parents=True, exist_ok=True)
        dialog_df.to_csv(output_path, index=False)
        print(f"\n✓ Saved to: {output_path.absolute()}")
        
    except Exception as e:
        print("❌ Error during processing:")
        print(f"   {type(e).__name__}: {str(e)}")

 DIALOG ACT ANALYSIS
Current working directory: /home/openclaw/.openclaw/workspace/projects/Employee performance 5/notebooks

❌ File not found
   Path tried: /home/openclaw/.openclaw/workspace/projects/Employee performance 5/notebooks/data/raw/sample_utterances.csv
   Common fixes:
     • Move file to data/raw/
     • Check spelling / hidden extensions (.csv.csv)
     • Run !pwd or print(Path.cwd()) to confirm directory
